In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [207]:
# 1. Carrega os dados processados
train_df = pd.read_csv('../dados/train_processado.csv')
test_df = pd.read_csv('../dados/test_processado.csv')

# 2. Seleciona as colunas que você definiu para o modelo
features = ['ClassePassageiro', 'Sexo', 'Idade', 'Total_Familiar', 'ValorBilhete_Pessoa']

X = train_df[features]
y = train_df['Sobreviveu']

X_teste = test_df[features].reindex(columns=features, fill_value=0)

# 3. Configura o StratifiedKFold (5 divisões mantendo a proporção de sobreviventes)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

previsoes_totais = np.zeros(len(X_teste))
acuracias_folds = []

# 4. Loop de treinamento com K-Folds
for fold, (treino_idx, val_idx) in enumerate(skf.split(X, y)):
    X_treino, y_treino = X.iloc[treino_idx], y.iloc[treino_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Instancia e treina a Random Forest com parâmetros conservadores
    modelo_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_split=5,
        random_state=42
    )
    modelo_rf.fit(X_treino, y_treino)
    
    # Valida no fold atual
    preds_val = modelo_rf.predict(X_val)
    acc = accuracy_score(y_val, preds_val)
    acuracias_folds.append(acc)
    print(f"Acurácia do Fold {fold + 1}: {acc * 100:.2f}%")
    
    # Acumula as previsões probabilísticas ou de votos para o teste final (Ensemble)
    previsoes_totais += modelo_rf.predict(X_teste) / skf.n_splits

print(f"\nAcurácia Média da Random Forest com K-Fold: {np.mean(acuracias_folds) * 100:.2f}%")

# 5. Define a premissa final (maioria dos votos dos 5 folds)
previsoes_finais = (previsoes_totais >= 0.5).astype(int)

# 6. Cria o arquivo de submissão
submissao = pd.DataFrame({
    'PassengerId': test_df['Id_Passageiro'],
    'Survived': previsoes_finais
})

submissao.to_csv('../dados/resultado_submission_rf_kfolds.csv', index=False)
print("Arquivo de submissão com Random Forest + K-Folds gerado com sucesso!")

Acurácia do Fold 1: 84.36%
Acurácia do Fold 2: 81.46%
Acurácia do Fold 3: 82.02%
Acurácia do Fold 4: 83.15%
Acurácia do Fold 5: 84.83%

Acurácia Média da Random Forest com K-Fold: 83.16%
Arquivo de submissão com Random Forest + K-Folds gerado com sucesso!


Testando melhores parâmetros

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

# 1. Carrega os dados processados
train_df = pd.read_csv('../dados/train_processado.csv')
test_df = pd.read_csv('../dados/test_processado.csv')

# 2. Seleciona as colunas que você definiu para o modelo
features = [
    'ClassePassageiro', 
    'Sexo', 
    'ValorBilhete_Pessoa', 
    'Titulo_Mr', 
    'Titulo_Mrs', 
    'Titulo_Miss', 
    'Sozinho'
]

X = train_df[features]
y = train_df['Sobreviveu']

X_teste = test_df[features].reindex(columns=features, fill_value=0)

# 3. Configura o StratifiedKFold (5 divisões mantendo a proporção de sobreviventes)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 4. Configuração do GridSearch para encontrar os melhores parâmetros
rf_base = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 5, 6],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1
)

# Executa a busca pelos melhores parâmetros
grid_search.fit(X, y)

print("Melhores parâmetros encontrados:", grid_search.best_params_)
print(f"Melhor acurácia na Validação Cruzada: {grid_search.best_score_ * 100:.2f}%\n")

# Pega o melhor modelo gerado pelo GridSearch
melhor_modelo = grid_search.best_estimator_

# 5. Loop de treinamento com K-Folds usando o modelo otimizado para gerar o Ensemble
previsoes_totais = np.zeros(len(X_teste))
acuracias_folds = []

for fold, (treino_idx, val_idx) in enumerate(skf.split(X, y)):
    X_treino, y_treino = X.iloc[treino_idx], y.iloc[treino_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Reinstancia o modelo com os melhores parâmetros do GridSearch
    modelo_rf = RandomForestClassifier(**grid_search.best_params_, random_state=42)
    modelo_rf.fit(X_treino, y_treino)
    
    # Valida no fold atual
    preds_val = modelo_rf.predict(X_val)
    acc = accuracy_score(y_val, preds_val)
    acuracias_folds.append(acc)
    print(f"Acurácia do Fold {fold + 1}: {acc * 100:.2f}%")
    
    # Acumula as previsões para o teste final (Ensemble)
    previsoes_totais += modelo_rf.predict(X_teste) / skf.n_splits

print(f"\nAcurácia Média da Random Forest Otimizada com K-Fold: {np.mean(acuracias_folds) * 100:.2f}%")

# 6. Define a premissa final (maioria dos votos dos 5 folds)
previsoes_finais = (previsoes_totais >= 0.5).astype(int)

# 7. Cria o arquivo de submissão
submissao = pd.DataFrame({
    'PassengerId': test_df['Id_Passageiro'],
    'Survived': previsoes_finais
})

submissao.to_csv('../dados/resultado_submission_rf_grid_kfolds.csv', index=False)
print("Arquivo de submissão com Random Forest Otimizada + K-Folds gerado com sucesso!")